In [35]:
import pandas as pd
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


# Pregunta A
El scraper cometió errores y guardó la misma app varias veces. Escriban una línea de código para identificar si hay nombres duplicados en la columna App (df.duplicated(subset=['App']).sum()). Si los hay, apliquen .drop_duplicates(subset=['App'], keep='first', inplace=True) para quedarse solo con la primera aparición de cada aplicación. ¿Cuántos registros duplicados purgaron?

In [36]:
#Identificar duplicados
df.duplicated(subset=['App']).sum()

antes = len(df)
df.drop_duplicates(subset=['App'], keep='first', inplace=True)

print(f"Registros antes:    {antes}")
print(f"Registros después:  {len(df)}")
print(f"Duplicados purgados: {antes - len(df)}")

Registros antes:    10841
Registros después:  9660
Duplicados purgados: 1181


## Respuesta
Se registraron 1,181 datos duplicados purgados. El dataset paso de 10841 a 9660

# Pregunta B
La columna Installs tiene texto como "1,000,000+". Un algoritmo no puede procesar eso.
Escriban una instrucción encadenando el método .str.replace() para eliminar el símbolo + y otro para eliminar la coma ,. (Ojo: utilicen regex=False o escapen el símbolo más \+).
Al final de la cadena, utilicen .astype(int) (o float si tienen NaNs) para convertir la columna.
Comprueben el éxito ejecutando df['Installs'].mean(). ¿Cuál es el promedio real de descargas?

In [37]:
df['Installs'].unique()[:10]

array(['10,000+', '500,000+', '5,000,000+', '50,000,000+', '100,000+',
       '50,000+', '1,000,000+', '10,000,000+', '5,000+', '100,000,000+'],
      dtype=object)

In [38]:
# Cadena de limpieza
df['Installs'] = pd.to_numeric(
    df['Installs']
      .str.replace('+', '', regex=False)
      .str.replace(',', '', regex=False),
    errors='coerce'
)

print("Filas que no se pudieron convertir:", df['Installs'].isna().sum())

Filas que no se pudieron convertir: 1


In [39]:
df = df.dropna(subset=['Installs']).copy()
df['Installs'] = df['Installs'].astype('int64')

df['Installs'].dtype

dtype('int64')

In [40]:
df['Installs'].mean()
print("Media:    {:,.2f}".format(df['Installs'].mean()))
print("Mediana:  {:,.0f}".format(df['Installs'].median()))

Media:    7,777,506.73
Mediana:  100,000


## Respuesta B
El promedio real de descargas es 7,777,506.73

# Pregunta C
La columna Price tiene valores como "$4.99" y "0". Apliquen la misma lógica arquitectónica que en la Pregunta B: eliminen el símbolo de dólar $ y conviertan la columna usando .astype(float).

In [41]:
df['Price'] = df['Price'].str.replace('$', '', regex=False).astype(float)

df['Price'].describe()

count    9659.000000
mean        1.099299
std        16.852152
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       400.000000
Name: Price, dtype: float64

## Respuesta C
La columna queda como float64

# Pregunta D
Ahora que el precio es numérico, ejecuten df['Price'].max(). Notarán que la app más cara cuesta alrededor de 400 dólares.
Usen un filtro booleano (df[df['Price'] > 200]) para descubrir cuáles son estas apps. Verán que son aplicaciones basura o de broma.
Si metemos estas apps al modelo, la red neuronal creerá que vender a 400 dólares es un modelo de negocio viable. Sobrescriban el DataFrame conservando únicamente las apps que cuesten menos de 50 dólares (df = df[df['Price'] < 50]).
Guarden su dataset final usando df.to_csv('playstore_limpio.csv', index=False).

In [42]:
df['Price'].max()
caras = df[df['Price'] > 200]
print("Apps con precio mayor a $200:", len(caras))
caras[['App', 'Category', 'Price', 'Installs', 'Rating']]

Apps con precio mayor a $200: 17


,App,Category,Price,Installs,Rating
4197,most expensive app (H),FAMILY,399.99,100,4.3
4362,💎 I'm rich,LIFESTYLE,399.99,10000,3.8
4367,I'm Rich - Trump Edition,LIFESTYLE,400.00,10000,3.6
5351,I am rich,LIFESTYLE,399.99,100000,3.8
5354,I am Rich Plus,FAMILY,399.99,10000,4.0
5355,I am rich VIP,LIFESTYLE,299.99,10000,3.8
5356,I Am Rich Premium,FINANCE,399.99,50000,4.1
5357,I am extremely Rich,LIFESTYLE,379.99,1000,2.9
5358,I am Rich!,FINANCE,399.99,1000,3.8
5359,I am rich(premium),FINANCE,399.99,5000,3.5


In [43]:
antes = len(df)
df = df[df['Price'] < 50]

print(f"Registros eliminados: {antes - len(df)}")
print(f"Registros finales:    {len(df)}")
print(f"Precio máximo ahora:  ${df['Price'].max()}")

Registros eliminados: 23
Registros finales:    9636
Precio máximo ahora:  $46.99


In [44]:
df.to_csv('playstore_limpio.csv', index=False)
print("Archivo guardado: playstore_limpio.csv")

Archivo guardado: playstore_limpio.csv
